# 00 · Data coverage (Phase 0)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arooj-zehra/pakistan-water-watch/blob/main/notebooks/00_coverage.ipynb)

Can we get **at least 5 years of daily Tarbela and Mangla inflow with under 10% of days missing**? Everything else in the project depends on the answer.

**How this notebook works on Colab:** the data is collected on your own computer and pushed to GitHub. This notebook clones the repo into the Colab session and reads it from there. So before opening it, run these on your computer and push the results:

```
python -m pipeline.probe_pmd
python -m pipeline.fetch_weather
python -m pipeline.coverage_report
git add data docs
git commit -m "Phase 0 data"
git push
```

Then in Colab: **Runtime → Run all**.

## Setup
Clones the repo (or pulls the latest version if this session already has it) and installs the requirements. Takes about a minute the first time.

In [ ]:
import os, sys

REPO_URL = "https://github.com/arooj-zehra/pakistan-water-watch.git"
REPO_DIR = "/content/pakistan-water-watch"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally: work from the repo root

sys.path.insert(0, os.getcwd())
!git log --oneline -1
print("Working in", os.getcwd())

## Verdict

In [ ]:
import pandas as pd
from pipeline import coverage_report as cr

if not cr.COVERAGE_CSV.exists():
    raise SystemExit("No data/inventory/pmd_coverage.csv in the repo yet. Run the probe on your "
                     "computer, commit and push (see the first cell), then Runtime > Run all.")

df = cr.load_coverage()
y = cr.yearly(df)
v = cr.verdict(y)
print(v.summary)

## Coverage per year

In [ ]:
y.style.format({c: "{:.0%}" for c in ["tarbela_missing", "mangla_missing", "worst_missing"]})

## Missing-days heatmap
Red = days with no inflow value, grey = month not probed.

In [ ]:
from IPython.display import Image
Image(filename=str(cr.plot_heatmap(df)))

## Download status per month
How each month's request ended (`ok`, `http_404`, network errors…), and any parser notes.

In [ ]:
print(df["status"].value_counts().to_string())
df[(~df.status.isin(["ok", "cached"])) | (df.problems.fillna("") != "")][
    ["month", "status", "tarbela_days", "mangla_days", "problems"]]

## All sources combined (PMD + news outlets)

The PMD archive alone is far too thin (only Sep-Dec 2015), so the real verdict comes from every source together: PMD plus the daily IRSA/WAPDA stories from APP and other news outlets. This rebuilds a temporary database inside the Colab session from the raw files committed to the repo (`data/raw/`), so nothing needs downloading unless a cache is incomplete.

The verdict is reported twice: as recorded, and with gaps of up to 3 days filled by linear interpolation (longer gaps stay empty).

In [ ]:
!python -m pipeline.backfill --pmd
!python -m pipeline.backfill_news

In [ ]:
dbr = cr.db_report()
if dbr is None:
    print("The database is empty: run probe_sources and backfill_app on your computer and push data/.")
else:
    section, v_raw, v_fill = dbr
    print(v_raw.summary)
    print(v_fill.summary)
    display(Image(filename=str(cr.DB_HEATMAP_PNG)))

### Parsed news figures
Every row links back to the article it came from, and the colour shows which outlet supplied it, so any odd value can be checked by eye.

In [ ]:
import matplotlib.pyplot as plt
from pipeline.backfill_news import PARSED_CSV, CONFLICTS_CSV
if PARSED_CSV.exists():
    news = pd.read_csv(PARSED_CSV, parse_dates=["date"])
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    for ax, sid in zip(axes, ("tarbela", "mangla")):
        for src, g in news[news.station_id == sid].groupby("source"):
            ax.plot(g["date"], g["inflow_cusecs"], ".", ms=3, label=src)
        ax.set_ylabel("inflow (cusecs)"); ax.set_title(sid.title()); ax.legend(fontsize=8, markerscale=3)
    plt.tight_layout(); plt.show()
    print("Days supplied by each outlet:")
    display(news.groupby(["source", "station_id"]).size().unstack())

### Where outlets disagree
Days on which two stories give inflows more than 1% apart. The kept value is the more complete story; open both links to see which is right.

In [ ]:
if CONFLICTS_CSV.exists():
    conflicts = pd.read_csv(CONFLICTS_CSV)
    print(len(conflicts), "disagreements")
    display(conflicts)

## Weather CSVs
One row per weather point. Any point marked `MISSING` needs `python -m pipeline.fetch_weather` re-run on your computer.

In [ ]:
from pipeline.config import WEATHER_DIR, WEATHER_POINTS

rows = []
for p in WEATHER_POINTS:
    path = WEATHER_DIR / f"era5_{p.id}.csv"
    if not path.exists():
        rows.append({"point": p.id, "catchment": p.catchment, "status": "MISSING"})
        continue
    w = pd.read_csv(path, parse_dates=["date"])
    rows.append({"point": p.id, "catchment": p.catchment, "status": "ok",
                 "first": w.date.min().date(), "last": w.date.max().date(),
                 "days": len(w), "missing_tmean": int(w.tmean_c.isna().sum())})
pd.DataFrame(rows)

## Weather sanity check
Mean temperature by catchment: there should be a clear seasonal cycle, with the upper Indus colder than the upper Jhelum.

In [ ]:
import matplotlib.pyplot as plt

frames = [pd.read_csv(p, parse_dates=["date"]) for p in sorted(WEATHER_DIR.glob("era5_*.csv"))]
if frames:
    w = pd.concat(frames)
    w["catchment"] = w.point_id.map({p.id: p.catchment for p in WEATHER_POINTS})
    monthly = w.set_index("date").groupby("catchment").tmean_c.resample("MS").mean().unstack(0)
    ax = monthly.plot(figsize=(11, 3.5), lw=1)
    ax.set_ylabel("monthly mean temperature (°C)")
    ax.set_title("ERA5 mean temperature, average over each catchment's points")
    plt.show()

## Decision

*Write the go / no-go call here once the numbers are in, and what you'll do about any gaps (see Risks in the project plan).*